In [10]:

import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")


Libraries imported successfully!


In [19]:
# Define project folder
project_dir = Path(
    r"C:\Users\bhava\OneDrive\Desktop\TED TALKS RECOMMADATION"
)

# Locate cleaned dataset
possible_files = [
    project_dir / "data" / "processed" / "ted_talks_cleaned.csv",
    project_dir / "data" / "processed_output" / "ted_talks_cleaned.csv"
]

file_path = next(
    (p for p in possible_files if p.is_file()),
    None
)

if file_path is None:
    raise FileNotFoundError(
        "Cleaned TED Talks dataset not found. "
        "Please check your data/processed folder."
    )

# Load dataset
df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())


Dataset loaded successfully!
Dataset shape: (3921, 24)


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,...,topics,related_talks,url,description,transcript,title_clean,description_clean,transcript_clean,topics_clean,combined_text
0,1,Al Gore sobre cómo evitar la crisis climática,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523396,2006-02-25,2006-06-27,TED2006,...,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,Con el mismo humor y humanidad que irradió en ...,Muchas gracias Chris. Y es en verdad un gran h...,al gore sobre c mo evitar la crisis clim tica,con el mismo humor y humanidad que irradi en u...,muchas gracias chris y es en verdad un gran ho...,alternative energy cars climate change culture...,al gore sobre c mo evitar la crisis clim tica ...
1,7,"David Pogue dice ""La Simplicidad Vende""",David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920803,2006-02-24,2006-06-27,TED2006,...,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,"El columnista del New York Times, David Pogue,...","Hola contestadora automática, mi vieja amiga. ...",david pogue dice la simplicidad vende,el columnista del new york times david pogue a...,hola contestadora autom tica mi vieja amiga ri...,computers entertainment interface design media...,david pogue dice la simplicidad vende el colum...
2,53,Un recorrido por la renovación urbana de la ma...,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664029,2006-02-26,2006-06-27,TED2006,...,"['MacArthur grant', 'activism', 'business', 'c...",{1041: '3 stories of local eco-entrepreneurshi...,https://www.ted.com/talks/majora_carter_greeni...,"En una charla altamente emotiva, la activista ...","Si están presentes aquí hoy, y estoy muy conte...",un recorrido por la renovaci n urbana de la ma...,en una charla altamente emotiva la activista g...,si est n presentes aqu hoy y estoy muy content...,macarthur grant activism business cities envir...,un recorrido por la renovaci n urbana de la ma...
3,66,Ken Robinson dice que las escuelas matan la cr...,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65052534,2006-02-25,2006-06-27,TED2006,...,"['children', 'creativity', 'culture', 'dance',...","{865: 'Bring on the learning revolution!', 173...",https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson plantea de manera entretenida...,"Buenos días. ¿Cómo están? Ha sido increíble, ¿...",ken robinson dice que las escuelas matan la cr...,sir ken robinson plantea de manera entretenida...,buenos d as c mo est n ha sido incre ble verda...,children creativity culture dance education pa...,ken robinson dice que las escuelas matan la cr...
4,92,Hans Rosling nos muestra las mejores estadísti...,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501766,2006-02-22,2006-06-27,TED2006,...,"['Africa', 'Asia', 'Google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,Una manera única de presentar datos. Con la en...,"Hace unos 10 años, emprendí la tarea de enseña...",hans rosling nos muestra las mejores estad sti...,una manera nica de presentar datos con la ener...,hace unos 10 a os emprend la tarea de ense ar ...,africa asia google demo economics global issue...,hans rosling nos muestra las mejores estad sti...


In [20]:

# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Available columns:")
print(df.columns.tolist())


Available columns:
['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript', 'title_clean', 'description_clean', 'transcript_clean', 'topics_clean', 'combined_text']


In [21]:

# Find title column
title_candidates = [
    "title",
    "talk_title",
    "talks_title",
    "talk_name",
    "name"
]

title_col = next(
    (col for col in title_candidates if col in df.columns),
    None
)

if title_col is None:
    raise ValueError(
        "Title column not found. Check your dataset columns."
    )

# Create combined_text if it does not exist
if "combined_text" not in df.columns:

    text_candidates = [
        "transcript",
        "description",
        "tags",
        "tags_text",
        "topics",
        "details",
        "summary"
    ]

    available_text_columns = [
        col for col in text_candidates if col in df.columns
    ]

    if not available_text_columns:
        raise ValueError(
            "No transcript, description, tags, or other text columns found."
        )

    df["combined_text"] = (
        df[available_text_columns]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
    )

# Prepare usable dataset
usable_df = df[[title_col, "combined_text"]].copy()

usable_df = usable_df.dropna(subset=[title_col])

usable_df[title_col] = usable_df[title_col].astype(str).str.strip()

usable_df["combined_text"] = (
    usable_df["combined_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Remove empty titles and duplicate titles
usable_df = usable_df[
    (usable_df[title_col] != "") &
    (usable_df[title_col].str.lower() != "nan")
]

usable_df = usable_df.drop_duplicates(
    subset=[title_col]
).reset_index(drop=True)

# Include the title in recommendation text
usable_df["recommendation_text"] = (
    usable_df[title_col] + " " + usable_df["combined_text"]
)

print("Dataset prepared successfully!")
print("Total usable TED Talks:", len(usable_df))

display(usable_df.head())

Dataset prepared successfully!
Total usable TED Talks: 3921


,title,combined_text,recommendation_text
0,Al Gore sobre cómo evitar la crisis climática,al gore sobre c mo evitar la crisis clim tica ...,Al Gore sobre cómo evitar la crisis climática ...
1,"David Pogue dice ""La Simplicidad Vende""",david pogue dice la simplicidad vende el colum...,"David Pogue dice ""La Simplicidad Vende"" david ..."
2,Un recorrido por la renovación urbana de la ma...,un recorrido por la renovaci n urbana de la ma...,Un recorrido por la renovación urbana de la ma...
3,Ken Robinson dice que las escuelas matan la cr...,ken robinson dice que las escuelas matan la cr...,Ken Robinson dice que las escuelas matan la cr...
4,Hans Rosling nos muestra las mejores estadísti...,hans rosling nos muestra las mejores estad sti...,Hans Rosling nos muestra las mejores estadísti...


In [22]:

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2)
)

# Convert text into numerical vectors
tfidf_matrix = tfidf.fit_transform(
    usable_df["recommendation_text"]
)

print("TF-IDF vectors generated successfully!")
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF vectors generated successfully!
TF-IDF matrix shape: (3921, 5000)


In [23]:
# Calculate cosine similarity
cosine_sim_matrix = cosine_similarity(tfidf_matrix)

print("Cosine similarity calculated successfully!")
print("Similarity matrix shape:", cosine_sim_matrix.shape)

Cosine similarity calculated successfully!
Similarity matrix shape: (3921, 3921)


In [24]:
def recommend_talks(title, top_n=5):

    # Validate top_n
    if not isinstance(top_n, int) or isinstance(top_n, bool) or top_n < 1:
        raise ValueError("top_n must be a positive integer.")

    # Check whether title exists
    matching_titles = usable_df[
        usable_df[title_col].str.lower() == title.strip().lower()
    ]

    # Handle invalid title
    if matching_titles.empty:

        available_titles = usable_df[title_col].tolist()

        suggestions = get_close_matches(
            title,
            available_titles,
            n=3,
            cutoff=0.4
        )

        if suggestions:
            raise ValueError(
                f"Ted Talk '{title}' not found. "
                f"Did you mean: {suggestions}?"
            )

        raise ValueError(
            f"Ted Talk '{title}' not found in the dataset."
        )

    # Get index of selected talk
    talk_index = matching_titles.index[0]

    # Get similarity scores
    similarity_scores = list(
        enumerate(cosine_sim_matrix[talk_index])
    )

    # Sort by similarity in descending order
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Exclude selected talk itself
    similarity_scores = [
        item for item in similarity_scores
        if item[0] != talk_index
    ]

    # Select top N recommendations
    similarity_scores = similarity_scores[:top_n]

    # Prepare results
    recommendations = []

    for index, score in similarity_scores:

        recommendations.append({
            "Recommended Talk": usable_df.iloc[index][title_col],
            "Similarity Score": round(float(score), 4)
        })

    # Return readable DataFrame
    return pd.DataFrame(recommendations)

In [25]:
# Display some available TED Talk titles
display(usable_df[[title_col]].head(10))

,title
0,Al Gore sobre cómo evitar la crisis climática
1,"David Pogue dice ""La Simplicidad Vende"""
2,Un recorrido por la renovación urbana de la ma...
3,Ken Robinson dice que las escuelas matan la cr...
4,Hans Rosling nos muestra las mejores estadísti...
5,Julia Sweeney habla sobre abandonar a Dios
6,Joshua Prince-Ramus habla sobre la biblioteca ...
7,Dan Dennett responde a Rick Warren
8,Rick Warren acerca de una vida con propósito.
9,Jehane Noujaim inspira un día mundial de cine


In [26]:
# Select first TED Talk
test_title = usable_df.iloc[0][title_col]

print("Selected TED Talk:", test_title)

# Generate top 5 recommendations
recommendations = recommend_talks(test_title, top_n=5)

display(recommendations)

Selected TED Talk: Al Gore sobre cómo evitar la crisis climática


,Recommended Talk,Similarity Score
0,Cómo podemos cambiar el rumbo del clima,0.6961
1,La convocatoria a la acción de Bono por África,0.6827
2,Chris Bliss: La comedia es traducción.,0.6742
3,"Nacionalismo vs. globalismo, la nueva división...",0.6703
4,Al Gore: Una nueva concepción de la crisis cli...,0.6682


In [27]:
# Select three different TED Talk titles
test_titles = usable_df[title_col].head(3).tolist()

for title in test_titles:

    print("\nSelected TED Talk:", title)

    result = recommend_talks(title, top_n=5)

    display(result)


Selected TED Talk: Al Gore sobre cómo evitar la crisis climática


,Recommended Talk,Similarity Score
0,Cómo podemos cambiar el rumbo del clima,0.6961
1,La convocatoria a la acción de Bono por África,0.6827
2,Chris Bliss: La comedia es traducción.,0.6742
3,"Nacionalismo vs. globalismo, la nueva división...",0.6703
4,Al Gore: Una nueva concepción de la crisis cli...,0.6682



Selected TED Talk: David Pogue dice "La Simplicidad Vende"


,Recommended Talk,Similarity Score
0,Diseño y descubrimiento,0.7357
1,Emily Levine: la teoría del todo de una timadora,0.7256
2,Peter Hirshberg acerca de la TV y la Web,0.7086
3,Dan Gilbert en nuestras expectativas erróneas,0.7039
4,La mente detrás de Linux,0.7006



Selected TED Talk: Un recorrido por la renovación urbana de la mano de Majora Carter.


,Recommended Talk,Similarity Score
0,Wade Davis visita culturas en peligro,0.7335
1,Majora Carter: 3 historias de eco-activismo local,0.7250
2,Samantha Power habla sobre un héroe complicado,0.7178
3,"Nacionalismo vs. globalismo, la nueva división...",0.7114
4,Chris Bliss: La comedia es traducción.,0.7095


In [29]:
from difflib import get_close_matches

print("Library imported successfully!")

Library imported successfully!


In [30]:
from difflib import get_close_matches
import pandas as pd

def recommend_talks(title, top_n=5):

    # Check input
    if not isinstance(title, str) or not title.strip():
        raise ValueError("Please enter a valid TED Talk title.")

    if not isinstance(top_n, int) or isinstance(top_n, bool) or top_n < 1:
        raise ValueError("top_n must be a positive integer.")

    # Search for the selected TED Talk
    matching_titles = usable_df[
        usable_df[title_col].str.lower() == title.strip().lower()
    ]

    # Handle invalid title
    if matching_titles.empty:

        available_titles = usable_df[title_col].tolist()

        suggestions = get_close_matches(
            title,
            available_titles,
            n=3,
            cutoff=0.4
        )

        if suggestions:
            raise ValueError(
                f"Ted Talk '{title}' not found. "
                f"Did you mean: {suggestions}?"
            )

        raise ValueError(
            f"Ted Talk '{title}' not found in the dataset."
        )

    # Get selected talk index
    talk_index = matching_titles.index[0]

    # Get similarity scores
    similarity_scores = list(
        enumerate(cosine_sim_matrix[talk_index])
    )

    # Sort in descending order
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Exclude selected talk
    similarity_scores = [
        item for item in similarity_scores
        if item[0] != talk_index
    ]

    # Get top N recommendations
    similarity_scores = similarity_scores[:top_n]

    # Create recommendation DataFrame
    recommendations = []

    for index, score in similarity_scores:

        recommendations.append({
            "Recommended Talk": usable_df.iloc[index][title_col],
            "Similarity Score": round(float(score), 4)
        })

    return pd.DataFrame(recommendations)

In [31]:
try:

    result = recommend_talks(
        "This Is An Invalid TED Talk Title",
        top_n=5
    )

    display(result)

except ValueError as e:

    print("Error handled successfully!")
    print(e)

Error handled successfully!
Ted Talk 'This Is An Invalid TED Talk Title' not found. Did you mean: ['Chris Anderson de WIRED habla del Long Tail', 'El asalto al tren']?


In [32]:
test_title = usable_df.iloc[0][title_col]

print("Selected TED Talk:", test_title)

result = recommend_talks(test_title, top_n=5)

display(result)

Selected TED Talk: Al Gore sobre cómo evitar la crisis climática


,Recommended Talk,Similarity Score
0,Cómo podemos cambiar el rumbo del clima,0.6961
1,La convocatoria a la acción de Bono por África,0.6827
2,Chris Bliss: La comedia es traducción.,0.6742
3,"Nacionalismo vs. globalismo, la nueva división...",0.6703
4,Al Gore: Una nueva concepción de la crisis cli...,0.6682


In [33]:
test_title = usable_df.iloc[0][title_col]

result = recommend_talks(test_title, top_n=5)

if test_title.lower() not in (
    result["Recommended Talk"].str.lower().tolist()
):
    print("PASS: Selected TED Talk is excluded.")
else:
    print("FAIL: Selected TED Talk appears in recommendations.")

PASS: Selected TED Talk is excluded.


In [34]:
all_results = []

for title in test_titles:

    result = recommend_talks(title, top_n=5)

    result.insert(0, "Selected Talk", title)

    all_results.append(result)

# Combine all recommendation results
final_recommendations = pd.concat(
    all_results,
    ignore_index=True
)

print("Final TED Talks Recommendation Results:")

display(final_recommendations)

Final TED Talks Recommendation Results:


,Selected Talk,Recommended Talk,Similarity Score
0,Al Gore sobre cómo evitar la crisis climática,Cómo podemos cambiar el rumbo del clima,0.6961
1,Al Gore sobre cómo evitar la crisis climática,La convocatoria a la acción de Bono por África,0.6827
2,Al Gore sobre cómo evitar la crisis climática,Chris Bliss: La comedia es traducción.,0.6742
3,Al Gore sobre cómo evitar la crisis climática,"Nacionalismo vs. globalismo, la nueva división...",0.6703
4,Al Gore sobre cómo evitar la crisis climática,Al Gore: Una nueva concepción de la crisis cli...,0.6682
5,"David Pogue dice ""La Simplicidad Vende""",Diseño y descubrimiento,0.7357
6,"David Pogue dice ""La Simplicidad Vende""",Emily Levine: la teoría del todo de una timadora,0.7256
7,"David Pogue dice ""La Simplicidad Vende""",Peter Hirshberg acerca de la TV y la Web,0.7086
8,"David Pogue dice ""La Simplicidad Vende""",Dan Gilbert en nuestras expectativas erróneas,0.7039
9,"David Pogue dice ""La Simplicidad Vende""",La mente detrás de Linux,0.7006


In [35]:
# Save recommendation results
output_file = project_dir / "ted_talk_recommendations.csv"

final_recommendations.to_csv(
    output_file,
    index=False
)

print("Recommendation results saved successfully!")
print("Saved location:", output_file)

Recommendation results saved successfully!
Saved location: C:\Users\bhava\OneDrive\Desktop\TED TALKS RECOMMADATION\ted_talk_recommendations.csv
